# Suffix Trie & Suffix Tree

In [1]:
%config InlineBackend.figure_format = "retina"

import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

warnings.filterwarnings("ignore", message=".*tight_layout.*")


def safe_tight_layout(fig=None):
    """Call tight_layout suppressing the aspect-ratio incompatibility warning."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        if fig is None:
            plt.tight_layout()
        else:
            fig.tight_layout()


plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 13,
    "figure.facecolor": "#fafafa",
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

COLORS = {
    "match": "#4CAF50",
    "mismatch": "#F44336",
    "current": "#FFC107",
    "skip": "#90CAF9",
    "default": "#E0E0E0",
    "pattern": "#BBDEFB",
    "border": "#7E57C2",
}


def draw_string_row(ax, y, string, label="", highlights=None, offset=0):
    """Draw a row of character boxes at vertical position y."""
    highlights = highlights or {}
    for i, ch in enumerate(string):
        color = highlights.get(i, COLORS["default"])
        rect = mpatches.FancyBboxPatch(
            (i + offset, y), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.2,
        )
        ax.add_patch(rect)
        ax.text(i + offset + 0.45, y + 0.45, ch,
                ha="center", va="center", fontsize=14, fontweight="bold")
    if label:
        ax.text(offset - 0.3, y + 0.45, label,
                ha="right", va="center", fontsize=12, color="#555")


def make_stepper(step_widget, label="Step"):
    """Create a prev/next/first/last button bar tied to an IntSlider (hidden)."""
    btn_first = widgets.Button(description="|<", layout=widgets.Layout(width="40px"))
    btn_prev  = widgets.Button(description="<", layout=widgets.Layout(width="40px"))
    btn_next  = widgets.Button(description=">", layout=widgets.Layout(width="40px"))
    btn_last  = widgets.Button(description=">|", layout=widgets.Layout(width="40px"))
    counter   = widgets.Label(value=f"{label}: {step_widget.value}/{step_widget.max}")

    def update_label(*_):
        counter.value = f"{label}: {step_widget.value}/{step_widget.max}"

    step_widget.observe(update_label, "value")
    step_widget.observe(update_label, "max")

    def on_first(_): step_widget.value = step_widget.min
    def on_prev(_):  step_widget.value = max(step_widget.min, step_widget.value - 1)
    def on_next(_):  step_widget.value = min(step_widget.max, step_widget.value + 1)
    def on_last(_):  step_widget.value = step_widget.max

    btn_first.on_click(on_first)
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_last.on_click(on_last)

    return widgets.HBox([btn_first, btn_prev, counter, btn_next, btn_last])


print("Helpers loaded (retina mode).")

Helpers loaded (retina mode).


## Suffix Trie & Suffix Tree

Build and visualize the suffix trie and its compacted suffix tree for any input string. Uses `networkx` + `matplotlib` for tree layout.

In [2]:
import networkx as nx
from networkx.drawing.nx_agraph import graphviz_layout


def build_suffix_trie_graph(T):
    """Build a suffix trie as a networkx DiGraph. Keep it small (|T| <= 8)."""
    G = nx.DiGraph()
    G.add_node("root", label="root")
    node_id = 0
    for i in range(len(T)):
        cur = "root"
        for j in range(i, len(T)):
            ch = T[j]
            nxt = None
            for _, neighbor, data in G.out_edges(cur, data=True):
                if data["label"] == ch:
                    nxt = neighbor
                    break
            if nxt is None:
                node_id += 1
                name = f"n{node_id}"
                is_leaf = j == len(T) - 1
                G.add_node(name, label=str(i) if is_leaf else "")
                G.add_edge(cur, name, label=ch)
                cur = name
            else:
                cur = nxt
    return G


def build_suffix_tree_graph(T):
    """Build a compacted suffix tree as a networkx DiGraph."""
    # naive: build trie then compact
    class Node:
        _id = 0
        def __init__(self):
            Node._id += 1
            self.id = Node._id
            self.children = {}  # char -> (label_string, child_node)
            self.suffix_start = None

    Node._id = 0
    root = Node()
    for i in range(len(T)):
        cur = root
        j = i
        while j < len(T):
            ch = T[j]
            if ch in cur.children:
                label, child = cur.children[ch]
                # find how far we match along this edge
                k = 0
                while k < len(label) and j + k < len(T) and label[k] == T[j + k]:
                    k += 1
                if k == len(label):
                    cur = child
                    j += k
                else:
                    # split
                    mid = Node()
                    mid.children[label[k]] = (label[k:], child)
                    new_leaf = Node()
                    new_leaf.suffix_start = i
                    mid.children[T[j + k]] = (T[j + k:], new_leaf)
                    cur.children[ch] = (label[:k], mid)
                    break
            else:
                new_leaf = Node()
                new_leaf.suffix_start = i
                cur.children[ch] = (T[j:], new_leaf)
                break
        else:
            cur.suffix_start = i

    # convert to networkx
    G = nx.DiGraph()

    def add_nodes(node, name="root"):
        is_leaf = len(node.children) == 0
        lbl = str(node.suffix_start) if is_leaf and node.suffix_start is not None else ""
        G.add_node(name, label=lbl, is_leaf=is_leaf)
        for ch in sorted(node.children):
            edge_label, child = node.children[ch]
            child_name = f"n{child.id}"
            add_nodes(child, child_name)
            G.add_edge(name, child_name, label=edge_label)

    add_nodes(root)
    return G


def draw_tree_graph(G, title=""):
    """Draw a tree graph with edge labels."""
    if len(G.nodes) > 80:
        print("String too long for visualization (max ~8 chars).")
        return

    fig, ax = plt.subplots(figsize=(max(len(G.nodes) * 0.8, 6), max(len(G.nodes) * 0.5, 4)))

    try:
        pos = graphviz_layout(G, prog="dot")
    except Exception:
        pos = nx.spring_layout(G, seed=42)

    # separate leaf / internal
    leaves = [n for n in G.nodes if G.nodes[n].get("is_leaf", False) or G.out_degree(n) == 0]
    internals = [n for n in G.nodes if n not in leaves]

    nx.draw_networkx_nodes(G, pos, nodelist=internals, ax=ax,
                           node_color="#BBDEFB", node_size=350, edgecolors="#555")
    nx.draw_networkx_nodes(G, pos, nodelist=leaves, ax=ax,
                           node_color="#C8E6C9", node_size=350, edgecolors="#555",
                           linewidths=2.0)

    # node labels
    node_labels = {n: G.nodes[n].get("label", "") for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels=node_labels, ax=ax, font_size=10, font_weight="bold")

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#777",
                           arrows=True, arrowsize=12, width=1.5)

    edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                                  font_size=11, font_color="#9C27B0", font_family="monospace")

    ax.set_title(title, fontsize=13, pad=12)
    ax.axis("off")
    safe_tight_layout(fig)
    plt.show()


T_tree_input = widgets.Text(value="cacao$", description="T:", layout=widgets.Layout(width="300px"))
tree_type = widgets.ToggleButtons(options=["Suffix Trie", "Suffix Tree"], description="View:")
out_tree = widgets.Output()


def refresh_tree(_=None):
    T = T_tree_input.value
    if not T or len(T) > 10:
        with out_tree:
            clear_output(wait=True)
            print("Keep |T| <= 10 for readable visualization.")
        return
    with out_tree:
        clear_output(wait=True)
        if tree_type.value == "Suffix Trie":
            G = build_suffix_trie_graph(T)
            draw_tree_graph(G, title=f"Suffix Trie of \"{T}\"")
        else:
            G = build_suffix_tree_graph(T)
            draw_tree_graph(G, title=f"Suffix Tree of \"{T}\"")


T_tree_input.observe(refresh_tree, "value")
tree_type.observe(refresh_tree, "value")
display(T_tree_input, tree_type, out_tree)
refresh_tree()

Text(value='cacao$', description='T:', layout=Layout(width='300px'))

ToggleButtons(description='View:', options=('Suffix Trie', 'Suffix Tree'), value='Suffix Trie')

Output()

## Online Construction — Phase by Phase

Watch how the suffix trie grows character by character (Ukkonen's online approach on the *trie*).
At each phase $i$, the algorithm appends character $t_i$ and walks the **boundary path**, adding new transitions until it reaches a state that already has a $t_i$-edge.

- **Green** nodes = new states created in this phase
- **Yellow** nodes = boundary path states visited
- **Blue** nodes = existing states (unchanged)

In [ ]:
def build_strie_phases(T):
    """
    Build suffix trie phase by phase (online).
    Returns a list of (phase_i, char, G, new_nodes, boundary_visited)
    where G is the networkx DiGraph after phase i.
    """
    G = nx.DiGraph()
    G.add_node("root", label="ε")
    # suffix links: node -> node
    suffix_link = {"root": "_bot"}
    phases = []
    node_counter = [0]

    def make_node(label=""):
        node_counter[0] += 1
        name = f"n{node_counter[0]}"
        G.add_node(name, label=label)
        return name

    def get_child(node, ch):
        for _, nb, data in G.out_edges(node, data=True):
            if data["label"] == ch:
                return nb
        return None

    # Track the deepest state (top of boundary path)
    deepest = "root"

    for i, ti in enumerate(T):
        new_nodes = []
        boundary_visited = []
        r = deepest
        oldr = None

        while r != "_bot":
            boundary_visited.append(r)
            child = get_child(r, ti)
            if child is not None:
                # Already has a t_i-transition — stop
                if oldr is not None:
                    suffix_link[oldr] = child
                break
            else:
                # Create new state and transition
                new_name = make_node(label="")
                G.add_edge(r, new_name, label=ti)
                new_nodes.append(new_name)
                if oldr is not None:
                    suffix_link[oldr] = new_name
                oldr = new_name
                # Follow suffix link
                r = suffix_link.get(r, "_bot")
        else:
            # Reached _bot (shouldn't happen if _bot has all transitions to root)
            if oldr is not None:
                suffix_link[oldr] = "root"

        # If we stopped because child existed
        if r != "_bot" and child is not None and oldr is not None:
            suffix_link[oldr] = child

        # New deepest = old deepest + t_i
        new_deepest = get_child(deepest, ti)
        if new_deepest is not None:
            deepest = new_deepest

        # Label leaves with suffix start positions
        # (a leaf is a node with no outgoing edges — but this changes as we build)

        phases.append({
            "phase": i,
            "char": ti,
            "prefix": T[:i+1],
            "G": G.copy(),
            "new_nodes": list(new_nodes),
            "boundary_visited": list(boundary_visited),
        })

    return phases


def draw_strie_phase(T, phase_idx):
    """Draw the suffix trie at a given construction phase."""
    if not T or len(T) > 8:
        print("Keep |T| ≤ 8 for readable visualization.")
        return

    phases = build_strie_phases(T)
    if not phases:
        return
    phase_idx = min(phase_idx, len(phases) - 1)
    phase = phases[phase_idx]
    G = phase["G"]

    fig, ax = plt.subplots(figsize=(max(len(G.nodes) * 0.8, 6),
                                     max(len(G.nodes) * 0.5, 4)))

    try:
        pos = graphviz_layout(G, prog="dot")
    except Exception:
        pos = nx.spring_layout(G, seed=42)

    # Color nodes
    node_colors = []
    for n in G.nodes:
        if n in phase["new_nodes"]:
            node_colors.append("#C8E6C9")  # green — new
        elif n in phase["boundary_visited"]:
            node_colors.append("#FFF9C4")  # yellow — boundary
        else:
            node_colors.append("#BBDEFB")  # blue — existing

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=300, edgecolors="#555", linewidths=1.2)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#777",
                           arrows=True, arrowsize=10, width=1.2)

    node_labels = {n: G.nodes[n].get("label", "") for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels=node_labels, ax=ax,
                            font_size=8, font_weight="bold")

    edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                                  font_size=10, font_color="#9C27B0",
                                  font_family="monospace")

    n_nodes = len(G.nodes)
    n_edges = len(G.edges)
    ax.set_title(
        f"Phase {phase['phase']+1}/{len(phases)}: "
        f"append '{phase['char']}' → \"{phase['prefix']}\"\n"
        f"Nodes: {n_nodes}  |  Edges: {n_edges}  |  "
        f"New states: {len(phase['new_nodes'])}  |  "
        f"Boundary visited: {len(phase['boundary_visited'])}",
        fontsize=10, pad=10)
    ax.axis("off")
    safe_tight_layout(fig)
    plt.show()


# ── Interactive widgets ──
T_phase_input = widgets.Text(value="cacao$", description="T:",
                              layout=widgets.Layout(width="300px"))
step_phase = widgets.IntSlider(value=0, min=0, max=0, description="Phase:",
                                layout=widgets.Layout(display="none"))


def _update_phase_max(*_):
    T = T_phase_input.value
    if T and len(T) <= 8:
        phases = build_strie_phases(T)
        step_phase.max = max(len(phases) - 1, 0)
        step_phase.value = min(step_phase.value, step_phase.max)


T_phase_input.observe(_update_phase_max, "value")
_update_phase_max()


def _draw_phase(T, step):
    if T and len(T) <= 8:
        draw_strie_phase(T, step)


out_phase = widgets.interactive_output(_draw_phase,
    {"T": T_phase_input, "step": step_phase})
stepper_phase = make_stepper(step_phase, "Phase")
display(T_phase_input, stepper_phase, out_phase)